# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [7]:
from data_cleaning import clean_data

In [10]:
import pandas as pd

# Load the three files
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)


# Clean column names BEFORE combining
for df in [df1, df2, df3]:
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

    if "st" in df.columns:
        df.rename(columns={"st": "state"}, inplace=True)


# Combine the three datasets
combined_df = pd.concat(
    [df1, df2, df3],
    ignore_index=True
)


# Clean gender
combined_df["gender"] = combined_df["gender"].replace({
    "F": "F",
    "female": "F",
    "Femal": "F",
    "M": "M",
    "Male": "M"
})


# Clean state
combined_df["state"] = combined_df["state"].replace({
    "AZ": "Arizona",
    "Cali": "California",
    "WA": "Washington"
})


# Clean education
combined_df["education"] = combined_df["education"].replace({
    "Bachelors": "Bachelor"
})


# Clean customer lifetime value
combined_df["customer_lifetime_value"] = (
    combined_df["customer_lifetime_value"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

combined_df["customer_lifetime_value"] = pd.to_numeric(
    combined_df["customer_lifetime_value"],
    errors="coerce"
)


# Clean vehicle class
combined_df["vehicle_class"] = combined_df["vehicle_class"].replace({
    "Sports Car": "Luxury",
    "Luxury SUV": "Luxury",
    "Luxury Car": "Luxury"
})


# Clean number of open complaints
complaints = combined_df["number_of_open_complaints"].astype(str)

middle_value = complaints.str.split("/").str[1]

combined_df["number_of_open_complaints"] = pd.to_numeric(
    middle_value.fillna(complaints),
    errors="coerce"
)


# Fill numeric null values with median
numeric_columns = combined_df.select_dtypes(include="number").columns

for column in numeric_columns:
    combined_df[column] = combined_df[column].fillna(
        combined_df[column].median()
    )


# Fill categorical null values with mode
categorical_columns = combined_df.select_dtypes(include="object").columns

for column in categorical_columns:
    mode_value = combined_df[column].mode()

    if not mode_value.empty:
        combined_df[column] = combined_df[column].fillna(
            mode_value.iloc[0]
        )


# Remove duplicates
combined_df = combined_df.drop_duplicates().reset_index(drop=True)


# Display cleaned combined data
print("Shape:", combined_df.shape)
print("\nNull values:")
print(combined_df.isnull().sum())

combined_df.head()

Shape: (9135, 11)

Null values:
customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,F,Master,7.714877e+03,0.0,1000.0,0.0,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,6.979536e+05,0.0,94.0,0.0,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1.288743e+06,48767.0,108.0,0.0,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,7.645862e+05,0.0,106.0,0.0,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,5.363077e+05,36357.0,68.0,0.0,Personal Auto,Four-Door Car,17.269323


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [11]:
marketing_df = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
)

marketing_df.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [12]:
sales_channel_revenue = pd.pivot_table(
    marketing_df,
    index="sales_channel",
    values="total_claim_amount",
    aggfunc="sum"
).round(2)

sales_channel_revenue

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [13]:
clv_by_gender_education = pd.pivot_table(
    marketing_df,
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

clv_by_gender_education

education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [14]:
complaints_by_policy_month = (
    marketing_df
    .groupby(["policy_type", "month"])["number_of_open_complaints"]
    .sum()
    .reset_index()
)

complaints_by_policy_month

,policy_type,month,number_of_open_complaints
0,Corporate Auto,1,443.434952
1,Corporate Auto,2,385.208135
2,Personal Auto,1,1727.605722
3,Personal Auto,2,1453.684441
4,Special Auto,1,87.074049
5,Special Auto,2,95.226817
